In [2]:
# Imports and setup
import sys
from pathlib import Path
# Ensure the repository `src` folder is on sys.path so `from utils...` works when running cells
sys.path.append(str(Path('../../src').resolve()))

import pandas as pd, numpy as np, time, traceback, ast
from pathlib import Path
from joblib import dump
from sklearn.base import clone
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from utils.models import MODELS
from utils.eval_metrics import evaluate_model

# Paths
BASE_DATA_PATH = Path("../../data/out/dataset_final.csv")
VALIDATION_PATH = Path("../../data/out/dataset_validation_final.csv")
BEST_MODELS = Path("../../data/out/best_models_ml/in/best_models_ml.xlsx")

MODELS_DIR = Path("../../data/out/best_models_ml/validation_ml/models")
PLOTS_DIR = Path("../../data/out/best_models_ml/validation_ml/plots")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "PRECIO"
categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

In [ ]:
def prepare_full_data(df, target_column='PRECIO', categorical_numeric=None):
    """
    Prepare X/y and the preprocessor using the entire dataset (no train/test split).
    This is used for training with 100% of the original dataset and then evaluating on an external validation set.
    """
    if categorical_numeric is None:
        categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

    # Identify numeric features (exclude categorical, target, and datetime column)
    numeric_features = [
        col for col in df.columns
        if col not in categorical_numeric and col != target_column and col != 'FECHA_HORA'
    ]

    # Build ColumnTransformer: scale numeric features, passthrough categorical
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', [c for c in categorical_numeric if c in df.columns])
    ])

    # Separate features and target
    X = df.drop(columns=['FECHA_HORA', target_column], errors='ignore')
    y = df[target_column]

    return preprocessor, X, y

In [ ]:
# Cargar datasets
df_train = pd.read_csv(BASE_DATA_PATH)
df_val = pd.read_csv(VALIDATION_PATH)

# Preprocesador ajustado con TODO el dataset original
preproc, X_train, _, y_train, _ = prepare_full_data(
    df_train, target_column=TARGET_COLUMN, categorical_numeric=categorical_numeric
)

# Transformar dataset de validación
X_val = preproc.transform(df_val.drop(columns=[TARGET_COLUMN]))
y_val = df_val[TARGET_COLUMN].values

# Leer mejores modelos
best_df = pd.read_excel(BEST_MODELS)

rows = []

for _, row in best_df.iterrows():
    variant = row["variant"]
    model_name = row["model"]
    config_params = ast.literal_eval(str(row.get("best_params", "{}")))

    try:
        base_estimator = MODELS[model_name]
        estimator = clone(base_estimator).set_params(**config_params)

        pipe = Pipeline([("preprocessor", preproc), ("model", estimator)])

        start = time.time()
        pipe.fit(X_train, y_train)   # ENTRENAR CON 100% DEL DATASET
        elapsed = time.time() - start

        y_pred = pipe.predict(X_val)
        metrics = evaluate_model(pipe, X_val, y_val)
        metrics.update({
            "variant": variant,
            "model": model_name,
            "train_time_s": elapsed,
            "n_train": len(y_train),
            "n_val": len(y_val),
        })
        rows.append(metrics)

        # Guardar modelo
        dump(pipe, MODELS_DIR / f"{variant}_{model_name}.joblib")

        # Graficar
        plt.figure(figsize=(12,5))
        plt.plot(y_val[:200], label="Real")
        plt.plot(y_pred[:200], label="Predicho")
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pred_{variant}_{model_name}.png")
        plt.close()

        plt.figure(figsize=(5,5))
        plt.scatter(y_val, y_pred, s=6, alpha=0.5)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', linewidth=2)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_{variant}_{model_name}.png")
        plt.close()

        print(f"OK {variant}-{model_name}: RMSE={metrics.get('RMSE'):.2f}")

    except Exception as e:
        print(f"ERROR {variant}-{model_name}: {e}")
        traceback.print_exc()

# Consolidar métricas
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel("../../data/out/best_models_ml/validation_ml/validation_ml_metrics.xlsx", index=False)

ValueError: not enough values to unpack (expected 5, got 3)